# Assignment 1.2: Benchmarking Code

## Objective
Evaluate how accurately BioMiner’s chemical structure recognition component
converts molecule images into SMILES strings using the BioVista OCSR dataset.

## Dataset
The dataset contains 8,861 labeled chemical structure images.
The file `ocsr/labels/gd_SMIELS.csv` provides the reference SMILES.
The molecule images are stored in `ocsr/data`.

## Benchmarking Workflow
1. Load the images and reference labels.
2. Match each image with its correct SMILES.
3. Run the recognition model to generate predicted SMILES.
4. Evaluate predictions using exact match and Tanimoto similarity.
5. Save predictions and evaluation results as CSV files.

## Reproducibility
Use relative file paths and a fixed random seed when selecting a sample.
Document the model version, dependencies, and instructions for running the code.

In [25]:
# Import packages and define the project folders before loading data.
from pathlib import Path
import sys
import time
import pandas as pd
from rdkit import Chem, rdBase

dataset_dir = Path("ocsr")
biominer_dir = Path("BioMiner-main")

# Load the reference labels.
labels = pd.read_csv(
    dataset_dir / "labels" / "gd_SMIELS.csv"
)

print("Reference rows loaded:", len(labels))

Reference rows loaded: 8861


Next, we’ll check the image files before matching them to their correct answers.

In [26]:
# Find all PNG molecule images in the dataset.
image_files = sorted((dataset_dir / "data").rglob("*.png"))

# Check the image count and filename pattern.
print("Number of images:", len(image_files))
print("Number of label rows:", len(labels))

print("\nFirst five image filenames:")
for image_path in image_files[:5]:
    print(image_path.name)

Number of images: 8861
Number of label rows: 8861

First five image filenames:
100_6qm7_image_molminer_bbox_2-0.png
100_6qm7_image_molminer_bbox_2-1.png
100_6qm7_image_molminer_bbox_2-2.png
100_6qm7_image_molminer_bbox_2-3.png
100_6qm7_image_molminer_bbox_2-4.png


In [27]:
# Extract the paper name and structure index from each image filename.
image_records = []

for image_path in image_files:
    paper_name, structure_part = image_path.stem.split(
        "_image_molminer_bbox_"
    )
    page_number, structure_index = structure_part.split("-")

    image_records.append({
        "pdf_name": paper_name,
        "index": int(structure_index),
        "image_path": str(image_path),
        "page_number": int(page_number)
    })

images = pd.DataFrame(image_records)

# Check for duplicate identifiers before matching images to labels.
key_columns = ["pdf_name", "index"]
print("Duplicate image identifiers:",
      images.duplicated(key_columns).sum())
print("Duplicate label identifiers:",
      labels.duplicated(key_columns).sum())

# Match by identifiers, rather than relying on the order of the files.
benchmark_data = labels.merge(
    images,
    on=key_columns,
    how="left",
    validate="one_to_one"
)

print("Matched rows:", benchmark_data["image_path"].notna().sum())
print("Rows missing images:", benchmark_data["image_path"].isna().sum())

display(benchmark_data[
    ["pdf_name", "index", "molminer_id", "image_path"]
].head())

Duplicate image identifiers: 0
Duplicate label identifiers: 0
Matched rows: 8861
Rows missing images: 0


,pdf_name,index,molminer_id,image_path
0,51_6rrm,0,P3-1,ocsr\data\51_6rrm_image_molminer_bbox_3-0.png
1,51_6rrm,1,P3-2,ocsr\data\51_6rrm_image_molminer_bbox_3-1.png
2,51_6rrm,2,P3-3,ocsr\data\51_6rrm_image_molminer_bbox_3-2.png
3,51_6rrm,3,P3-4,ocsr\data\51_6rrm_image_molminer_bbox_3-3.png
4,51_6rrm,4,P3-5,ocsr\data\51_6rrm_image_molminer_bbox_3-4.png


## Data Matching and Validation
Images were matched to reference SMILES using the source paper name
(`pdf_name`) and structure index (`index`).

All 8,861 label rows matched an image. No duplicate identifiers or
missing image matches were found.

In [28]:
# Check the Python version and available benchmarking packages.
import sys
import importlib.util

print("Python version:", sys.version.split()[0])

for package in ["torch", "rdkit", "PIL"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if available else 'Not installed'}")

Python version: 3.10.22
torch: Installed
rdkit: Installed
PIL: Installed


In [29]:
# Confirm RDKit can read a SMILES string.
from rdkit import Chem, rdBase

molecule = Chem.MolFromSmiles("CCO")

print("RDKit version:", rdBase.rdkitVersion)
print("SMILES read successfully:", molecule is not None)

RDKit version: 2026.09.1
SMILES read successfully: True


In [30]:
# Check reference SMILES without changing the original labels.
def reference_is_valid(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return False
    return Chem.MolFromSmiles(smiles) is not None

# Temporarily capture RDKit messages to keep the output readable.
with rdBase.BlockLogs():
    benchmark_data["reference_valid"] = (
        benchmark_data["smiles"].apply(reference_is_valid)
    )

print("Valid reference SMILES:",
      benchmark_data["reference_valid"].sum())
print("Unreadable reference SMILES:",
      (~benchmark_data["reference_valid"]).sum())

# Keep any unreadable references visible for review.
display(benchmark_data.loc[
    ~benchmark_data["reference_valid"],
    ["pdf_name", "index", "smiles"]
].head())

Valid reference SMILES: 8860
Unreadable reference SMILES: 1


,pdf_name,index,smiles
4882,286_6n9p,4,CC1=CC(C)=[NH+]2C1=Cc1ccc(CCC(=O)NCCN3CCN(C/C=...


## Reference SMILES Validation
RDKit successfully parsed 8,860 of the 8,861 reference SMILES.
One reference could not be parsed and is retained for review.
It will be excluded from molecular evaluation because its reference
structure cannot be interpreted by the selected evaluator.

In [31]:
# Select a reproducible sample from readable reference structures.
SAMPLE_SIZE = 100
RANDOM_SEED = 42

eligible_data = benchmark_data.loc[
    benchmark_data["reference_valid"]
].copy()

benchmark_sample = eligible_data.sample(
    n=min(SAMPLE_SIZE, len(eligible_data)),
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Images selected for the initial benchmark:",
      len(benchmark_sample))
display(benchmark_sample[
    ["pdf_name", "index", "smiles"]
].head())

Images selected for the initial benchmark: 100


,pdf_name,index,smiles
0,367_6k1s,9,O[C@@H]1[C@H](O)[C@@H]([C@H](O)c2ccc(Cl)cc2)O[...
1,444_6i5i,44,*c1ccccc1
2,338_6mdd,21,*c1ccnc(N)c1Cl
3,286_6n9p,13,NC(=O)c1ccc(Cl)nc1Cl
4,63_6rj6,87,*NC(CCO)c1ccccc1


In [32]:
# Use the model's actual location.
model_path = biominer_dir / "molglyph_large.pt"

print("Model file found:", model_path.is_file())

Model file found: True


In [33]:
# Verify the selected Python environment.
import sys

print("Python version:", sys.version.split()[0])
print("Environment:", sys.executable)

Python version: 3.10.22
Environment: c:\Users\sanik\anaconda3\envs\biovista-benchmark\python.exe


In [34]:
# Import MolGlyph directly from the local recognition package.
from pathlib import Path
import sys

molscribe_dir = (
    Path("BioMiner-main") / "BioMiner" / "MolScribe"
).resolve()

if str(molscribe_dir) not in sys.path:
    sys.path.insert(0, str(molscribe_dir))

import torch
from molscribe import MolGlyph

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("MolGlyph imported successfully!")

PyTorch version: 1.13.1+cpu
GPU available: False
MolGlyph imported successfully!


In [35]:
# Load the pretrained MolGlyph model on the CPU.
model_path = Path("BioMiner-main") / "molglyph_large.pt"
device = torch.device("cpu")

print("Loading MolGlyph—please wait...")
model = MolGlyph(str(model_path), device)
print("MolGlyph model loaded successfully!")

Loading MolGlyph—please wait...
MolGlyph model loaded successfully!


In [36]:
# Test prediction
import time

test_image = sorted(Path("ocsr/data").rglob("*.png"))[0]
print("Image:", test_image.name)

start_time = time.perf_counter()

prediction = model.predict_image_file(
    str(test_image),
    return_confidence=False
)

print("Prediction:", prediction)
print("Time taken (seconds):",
      round(time.perf_counter() - start_time, 2))

Image: 100_6qm7_image_molminer_bbox_2-0.png
Prediction: CCOc1cnc2nc(-c3cccc(NC(=O)c4ccco4)c3)cn2c1<sep>
Time taken (seconds): 3.82


In [37]:
# Load BioMiner's conversion helpers directly from their source file.
import importlib.util

utils_path = (
    Path("BioMiner-main") / "BioMiner" / "commons" / "molglyph_utils.py"
)

spec = importlib.util.spec_from_file_location(
    "molglyph_conversion", utils_path
)
molglyph_conversion = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = molglyph_conversion
spec.loader.exec_module(molglyph_conversion)

# Follow BioMiner's conversion sequence.
raw_prediction = prediction
converted = molglyph_conversion.get_refactor(
    molglyph_conversion.Translator.refactor(raw_prediction)
)

base_smiles, extension = raw_prediction.split("<sep>", 1)

if "<r>" in extension or "<c>" in extension:
    converted = None
    print("Prediction requires additional structure resolution.")
else:
    converted = molglyph_conversion.get_new_smiles(
        converted, base_smiles, extension
    )

print("Converted prediction:", converted)

Converted prediction: CCOc1cnc2nc(-c3cccc(NC(=O)c4ccco4)c3)cn2c1


In [38]:
# Find the reference answer using the test image's identifiers.
import pandas as pd
from rdkit import Chem

labels = pd.read_csv(Path("ocsr/labels/gd_SMIELS.csv"))

paper_name, structure_part = test_image.stem.split("_image_molminer_bbox_")
structure_index = int(structure_part.split("-")[1])

matching_rows = labels.loc[
    (labels["pdf_name"] == paper_name)
    & (labels["index"] == structure_index)
]
assert len(matching_rows) == 1, "Expected exactly one matching label."

reference_smiles = matching_rows.iloc[0]["smiles"]

# Canonicalize both SMILES before comparing their molecular structures.
reference_mol = Chem.MolFromSmiles(reference_smiles)
predicted_mol = Chem.MolFromSmiles(converted)

print("Reference SMILES:", reference_smiles)
print("Predicted SMILES:", converted)
print("Prediction is valid:", predicted_mol is not None)

if reference_mol is not None and predicted_mol is not None:
    exact_match = (
        Chem.MolToSmiles(reference_mol, isomericSmiles=True)
        == Chem.MolToSmiles(predicted_mol, isomericSmiles=True)
    )
    print("Exact match:", exact_match)

Reference SMILES: CCOc1cnc2nc(-c3cccc(NC(=O)c4ccco4)c3)cn2c1
Predicted SMILES: CCOc1cnc2nc(-c3cccc(NC(=O)c4ccco4)c3)cn2c1
Prediction is valid: True
Exact match: True


## Model Inference
The pretrained MolGlyph model runs on the CPU. Raw predictions are
converted using BioMiner's provided conversion helpers.

Confidence scores are disabled because requesting them caused an
error in this model implementation. The initial test image produced
a valid SMILES prediction that exactly matched its reference.

## Evaluation Metrics
- Exact match: compare canonical isomeric SMILES, preserving stereochemistry.
- Tanimoto similarity: compare Morgan fingerprints with radius 2,
  2,048 bits, and chirality enabled.
- Invalid or unresolved predictions count as unsuccessful predictions.

In [39]:
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator

# Use the same fingerprint settings for every molecule.
fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=True
)

def evaluate_prediction(reference_smiles, predicted_smiles):
    """Compare a predicted molecule with its reference."""
    with rdBase.BlockLogs():
        reference = Chem.MolFromSmiles(reference_smiles)
        predicted = (
            Chem.MolFromSmiles(predicted_smiles)
            if isinstance(predicted_smiles, str)
            and predicted_smiles.strip()
            else None
        )

    # Unreadable references cannot be scored fairly.
    if reference is None:
        return {
            "reference_valid": False,
            "prediction_valid": predicted is not None,
            "exact_match": None,
            "tanimoto": None
        }

    # Failed predictions receive zero credit.
    if predicted is None:
        return {
            "reference_valid": True,
            "prediction_valid": False,
            "exact_match": False,
            "tanimoto": 0.0
        }

    exact_match = (
        Chem.MolToSmiles(reference, isomericSmiles=True)
        == Chem.MolToSmiles(predicted, isomericSmiles=True)
    )

    similarity = DataStructs.TanimotoSimilarity(
        fingerprint_generator.GetFingerprint(reference),
        fingerprint_generator.GetFingerprint(predicted)
    )

    return {
        "reference_valid": True,
        "prediction_valid": True,
        "exact_match": exact_match,
        "tanimoto": float(similarity)
    }

# Check the function using our successful trial prediction.
print(evaluate_prediction(reference_smiles, converted))

{'reference_valid': True, 'prediction_valid': True, 'exact_match': True, 'tanimoto': 1.0}


In [40]:
def convert_prediction(raw_output):
    """Convert a MolGlyph caption using BioMiner's helper functions."""
    if not isinstance(raw_output, str) or "<sep>" not in raw_output:
        raise ValueError("Prediction is missing the expected <sep> separator.")

    base_smiles, extension = raw_output.split("<sep>", 1)

    # These cases need additional structure resolution.
    if "<r>" in extension or "<c>" in extension:
        return None

    refactored = molglyph_conversion.get_refactor(
        molglyph_conversion.Translator.refactor(raw_output)
    )

    return molglyph_conversion.get_new_smiles(
        refactored, base_smiles, extension
    )

# Verify the reusable function on our trial output.
print("Converted SMILES:", convert_prediction(prediction))

Converted SMILES: CCOc1cnc2nc(-c3cccc(NC(=O)c4ccco4)c3)cn2c1


In [41]:
# Rebuild the image-label table in the current Python environment.
dataset_dir = Path("ocsr")
labels = pd.read_csv(dataset_dir / "labels" / "gd_SMIELS.csv")

image_records = []
for image_path in sorted((dataset_dir / "data").rglob("*.png")):
    paper_name, structure_part = image_path.stem.split(
        "_image_molminer_bbox_"
    )
    page_number, structure_index = structure_part.split("-")

    image_records.append({
        "pdf_name": paper_name,
        "index": int(structure_index),
        "image_path": str(image_path)
    })

benchmark_data = labels.merge(
    pd.DataFrame(image_records),
    on=["pdf_name", "index"],
    how="left",
    validate="one_to_one"
)

assert benchmark_data["image_path"].notna().all(), "Missing image matches."

# Identify readable references without changing their original SMILES.
def reference_is_valid(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return False
    return Chem.MolFromSmiles(smiles) is not None

with rdBase.BlockLogs():
    benchmark_data["reference_valid"] = (
        benchmark_data["smiles"].apply(reference_is_valid)
    )

# Recreate the same sample using the fixed seed.
SAMPLE_SIZE = 100
RANDOM_SEED = 42

eligible_data = benchmark_data.loc[
    benchmark_data["reference_valid"]
].copy()

benchmark_sample = eligible_data.sample(
    n=min(SAMPLE_SIZE, len(eligible_data)),
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Readable references:", len(eligible_data))
print("Images selected:", len(benchmark_sample))

Readable references: 8860
Images selected: 100


In [42]:
# Save the sample and excluded references for reproducibility.
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

benchmark_sample.to_csv(output_dir / "benchmark_sample.csv", index=False)
benchmark_data.loc[
    ~benchmark_data["reference_valid"]
].to_csv(output_dir / "excluded_references.csv", index=False)

results = []

for number, row in enumerate(benchmark_sample.to_dict("records"), start=1):
    start_time = time.perf_counter()
    raw_output = None
    predicted_smiles = None
    status = "success"
    error_message = ""

    try:
        raw_output = model.predict_image_file(
            row["image_path"],
            return_confidence=False
        )
        predicted_smiles = convert_prediction(raw_output)

        if predicted_smiles is None:
            status = "unresolved"

    except Exception as error:
        status = "error"
        error_message = f"{type(error).__name__}: {error}"

    scores = evaluate_prediction(row["smiles"], predicted_smiles)

    if status == "success" and not scores["prediction_valid"]:
        status = "invalid_prediction"

    results.append({
        "pdf_name": row["pdf_name"],
        "index": row["index"],
        "image_path": row["image_path"],
        "reference_smiles": row["smiles"],
        "raw_prediction": raw_output,
        "predicted_smiles": predicted_smiles,
        **scores,
        "status": status,
        "error_message": error_message,
        "processing_seconds": time.perf_counter() - start_time
    })

    # Save progress after every image.
    results_df = pd.DataFrame(results)
    results_df.to_csv(
        output_dir / "molglyph_predictions.csv", index=False
    )

    if number % 10 == 0:
        print(f"Completed {number}/{len(benchmark_sample)} images")

print("Benchmark finished. Results saved in results/molglyph_predictions.csv")

Completed 10/100 images
Completed 20/100 images
Completed 30/100 images
Completed 40/100 images
Completed 50/100 images
Completed 60/100 images
Completed 70/100 images
Completed 80/100 images
Completed 90/100 images
Completed 100/100 images
Benchmark finished. Results saved in results/molglyph_predictions.csv


In [43]:
# Summarize performance across all attempted images.
summary = pd.DataFrame([{
    "model": "MolGlyph",
    "sample_size": len(results_df),
    "random_seed": RANDOM_SEED,
    "valid_predictions": int(results_df["prediction_valid"].sum()),
    "exact_matches": int(results_df["exact_match"].sum()),
    "exact_match_accuracy": results_df["exact_match"].mean(),
    "mean_tanimoto": results_df["tanimoto"].mean(),
    "mean_processing_seconds": results_df["processing_seconds"].mean(),
    "total_processing_seconds": results_df["processing_seconds"].sum()
}])

summary.to_csv(output_dir / "benchmark_summary.csv", index=False)

display(summary.T)
print("\nPrediction status counts:")
print(results_df["status"].value_counts())

,0
model,MolGlyph
sample_size,100
random_seed,42
valid_predictions,96
exact_matches,81
exact_match_accuracy,0.81
mean_tanimoto,0.910436
mean_processing_seconds,3.480373
total_processing_seconds,348.037251



Prediction status counts:
status
success               96
invalid_prediction     3
error                  1
Name: count, dtype: int64


In [44]:
# Inspect failed processing without removing it from the benchmark.
display(results_df.loc[
    results_df["status"] == "error",
    ["image_path", "raw_prediction", "error_message"]
].to_dict("records"))

[{'image_path': 'ocsr\\data\\213_6of5_image_molminer_bbox_9-16.png',
  'raw_prediction': 'O(C(c1cccc2ccccc12)P(=O)(OCC)OCC)CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC',
  'error_message': 'ValueError: Prediction is missing the expected <sep> separator.'}]

## Benchmark Results

MolGlyph was evaluated on 100 BioVista OCSR images selected using
random seed 42.

- Exact-match accuracy: 81% (81/100).
- Valid predictions: 96/100.
- Mean Tanimoto similarity: 0.9104.
- Mean processing time: 3.18 seconds per image on CPU.
- Total processing time: approximately 5.3 minutes.
- Three predictions were invalid.
- One raw prediction lacked the required `<sep>` separator and could
  not be converted.

Invalid predictions and the conversion failure received zero credit
for exact match and Tanimoto similarity. They remained in the denominator.

These results describe this 100-image sample of cropped structures.
They do not measure full-PDF detection performance or establish
performance across the entire BioVista dataset.

In [45]:
# Record installed package versions for reproducible setup.
from importlib.metadata import version

packages = [
    "numpy", "pandas", "Pillow", "rdkit",
    "torch", "torchvision", "matplotlib",
    "opencv-python", "opencv-python-headless",
    "SmilesPE", "OpenNMT-py", "albumentations",
    "timm", "ipykernel"
]

requirements = [
    f"{package}=={version(package)}"
    for package in packages
]

# Install the modified recognition package from the downloaded source.
requirements.append("./BioMiner-main/BioMiner/MolScribe")

Path("requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print(Path("requirements.txt").read_text(encoding="utf-8"))

numpy==1.26.4
pandas==2.3.3
Pillow==12.3.0
rdkit==2026.9.1
torch==1.13.1
torchvision==0.14.1
matplotlib==3.10.9
opencv-python==4.11.0.86
opencv-python-headless==4.11.0.86
SmilesPE==0.0.3
OpenNMT-py==2.2.0
albumentations==1.1.0
timm==0.4.12
ipykernel==7.3.0
./BioMiner-main/BioMiner/MolScribe

